# Advanced RAG: Multi-Document Conversational Pipeline

This notebook extends the Basic RAG pipeline with:
- **Multi-format document ingestion** (TXT, PDF, MD)
- **Conversational memory** using chat_history
- **History-aware retriever** that reformulates follow-up questions into standalone queries
- **Source attribution** for traceability

**Architecture**: Multi-Loader → Unified Splitting → Chroma Indexing → History-Aware Retriever → Conversational Retrieval Chain

## Step 1: Multi-Format Document Loading

**Why multiple loaders?** Real-world RAG systems ingest heterogeneous data sources. Each loader handles format-specific parsing:
- TextLoader: Plain text (.txt)
- PyPDFLoader: PDF files with page-level splitting (.pdf)
- TextLoader for Markdown: Preserves raw markdown content (.md)

**Common Mistake**: Not handling different encodings or empty documents from loaders.

In [1]:
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import TextLoader, PyPDFLoader

load_dotenv(find_dotenv())

# Load speech.txt
txt_loader = TextLoader("../documents/speech.txt", encoding="utf-8")
txt_docs = txt_loader.load()
print(f"speech.txt: {len(txt_docs)} doc(s), {len(txt_docs[0].page_content)} chars")

# Load attention.pdf (the famous "Attention Is All You Need" paper)
pdf_loader = PyPDFLoader("../documents/attention.pdf")
pdf_docs = pdf_loader.load()
print(f"attention.pdf: {len(pdf_docs)} page(s), total {sum(len(d.page_content) for d in pdf_docs)} chars")

# Load sample_docs.md
md_loader = TextLoader("../documents/sample_docs.md", encoding="utf-8")
md_docs = md_loader.load()
print(f"sample_docs.md: {len(md_docs)} doc(s), {len(md_docs[0].page_content)} chars")

# Combine all documents into a unified list
all_documents = txt_docs + pdf_docs + md_docs
print(f"\nTotal documents loaded: {len(all_documents)}")


/var/folders/0h/sdy0_vy9385bh841gfp_jzdm0000gn/T/ipykernel_49480/3724128982.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader
/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


speech.txt: 1 doc(s), 3624 chars
attention.pdf: 15 page(s), total 39597 chars
sample_docs.md: 1 doc(s), 260 chars

Total documents loaded: 17


## Step 2: Chunking with Metadata Preservation

**Key Design Decision**: The splitter preserves source metadata (filename, page number) from the original Document objects. This is critical for source attribution in production RAG.

**Common Mistake**: Creating new Document objects manually without carrying forward the original metadata.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

all_chunks = text_splitter.split_documents(all_documents)

print(f"Total chunks after splitting: {len(all_chunks)}")

# Show chunk distribution by source
from collections import Counter
source_counts = Counter()
for chunk in all_chunks:
    source = chunk.metadata.get("source", "unknown")
    source_counts[source.split("/")[-1]] += 1

print("\nChunks per source:")
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")


Total chunks after splitting: 104

Chunks per source:
  speech.txt: 10 chunks
  attention.pdf: 93 chunks
  sample_docs.md: 1 chunks


## Step 3: Vector Store Indexing

All chunks from all sources are indexed into a single Chroma collection. The metadata (source file, page number) is preserved for retrieval-time source attribution.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Create in-memory Chroma vector store
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Indexed {len(all_chunks)} chunks into Chroma. Retriever configured with k=4.")


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 8.046457685s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', '

## Step 4: History-Aware Retriever

**The Problem**: In multi-turn conversations, follow-up questions like "Tell me more about that" or "What about its performance?" lack context without chat history.

**The Solution**: create_history_aware_retriever uses an LLM to reformulate the latest user question into a standalone search query by incorporating chat history context.

**How It Works**:
1. Takes chat_history + latest question.
2. LLM reformulates: "Tell me more about that" → "Tell me more about the Transformer self-attention mechanism."
3. Reformulated query is passed to the base retriever for vector search.

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# Contextualization prompt: reformulates follow-up questions into standalone queries
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", """Given a chat history and the latest user question which might reference context in the chat history,
formulate a standalone question which can be understood without the chat history.
Do NOT answer the question, just reformulate it if needed and otherwise return it as is."""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# Create history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

print("History-aware retriever created successfully!")


NameError: name 'retriever' is not defined

## Step 5: Conversational RAG Chain

**Full Architecture**:
1. User asks a question with chat_history.
2. History-aware retriever reformulates the question and retrieves relevant chunks.
3. Stuff documents chain inserts retrieved context into the system prompt.
4. LLM generates an answer grounded in the retrieved context.
5. Answer and source documents are returned.

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

# QA prompt with system instruction for source-grounded answering
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert assistant that answers questions based on the provided context.
Use ONLY the provided context to answer. If the context does not contain the answer, say:
"I don't have enough information to answer this question."

When possible, mention which source document the information comes from.

Context:
{context}"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# Create the stuff documents chain
qa_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create the full conversational retrieval chain
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

print("Conversational RAG chain built successfully!")
print(f"Chain type: {type(conversational_rag_chain).__name__}")


NameError: name 'history_aware_retriever' is not defined

## Step 6: Multi-Turn Conversation Demo

This demonstrates a 3-turn conversation where each follow-up question depends on the previous context. The history-aware retriever automatically reformulates follow-up questions.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

# --- Turn 1 ---
question_1 = "What is the Transformer architecture about?"
result_1 = conversational_rag_chain.invoke({
    "input": question_1,
    "chat_history": chat_history
})

print("=" * 60)
print(f"Turn 1 Question: {question_1}")
print(f"Answer: {result_1['answer']}")
print(f"Sources: {len(result_1['context'])} chunks retrieved")
for doc in result_1['context']:
    src = doc.metadata.get('source', 'unknown').split('/')[-1]
    page = doc.metadata.get('page', '')
    print(f"  - {src}{f' (page {page})' if page != '' else ''}")

# Update chat history
chat_history.extend([
    HumanMessage(content=question_1),
    AIMessage(content=result_1["answer"])
])


NameError: name 'conversational_rag_chain' is not defined

In [ ]:
# --- Turn 2 (Follow-up question referencing Turn 1) ---
question_2 = "What attention mechanism does it use?"
result_2 = conversational_rag_chain.invoke({
    "input": question_2,
    "chat_history": chat_history
})

print("=" * 60)
print(f"Turn 2 Question: {question_2}")
print(f"(History-aware retriever reformulates this into a standalone query)")
print(f"Answer: {result_2['answer']}")

# Update chat history
chat_history.extend([
    HumanMessage(content=question_2),
    AIMessage(content=result_2["answer"])
])


NameError: name 'conversational_rag_chain' is not defined

In [ ]:
# --- Turn 3 (Cross-document question) ---
question_3 = "Now tell me about the speech document - what values does it advocate for?"
result_3 = conversational_rag_chain.invoke({
    "input": question_3,
    "chat_history": chat_history
})

print("=" * 60)
print(f"Turn 3 Question: {question_3}")
print(f"Answer: {result_3['answer']}")
print(f"\nSources: {len(result_3['context'])} chunks retrieved")
for doc in result_3['context']:
    src = doc.metadata.get('source', 'unknown').split('/')[-1]
    page = doc.metadata.get('page', '')
    print(f"  - {src}{f' (page {page})' if page != '' else ''}")


NameError: name 'conversational_rag_chain' is not defined

## Step 7: Source Traceability

Production RAG systems must show which documents contributed to the answer. The retrieval chain returns the full Document objects in result["context"], including all metadata from the original loaders.

In [ ]:
# Demonstrate full source traceability
trace_result = conversational_rag_chain.invoke({
    "input": "What is the relationship between democracy and war in the speech?",
    "chat_history": []
})

print("=== Source Traceability Demo ===")
print(f"Question: {trace_result['input']}")
print(f"Answer: {trace_result['answer']}")
print(f"\n--- Retrieved Context Details ---")
for i, doc in enumerate(trace_result['context'], 1):
    print(f"\nChunk {i}:")
    print(f"  Source: {doc.metadata.get('source', 'N/A')}")
    print(f"  Page: {doc.metadata.get('page', 'N/A')}")
    print(f"  Content Preview: {doc.page_content[:120]}...")


NameError: name 'conversational_rag_chain' is not defined